In [ ]:
import pickle
import socket
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

np.random.seed(0)
torch.manual_seed(0)

sys.path.insert(0, str(Path.cwd().parent))

from fig_utils.plots import plot_metric_by_rank
from fig_utils.decoding import within_session_decoding_acc
from fig_utils.spike_stats import per_session_delay_tuning
from vi_rnn.datasets import SWM_dataset_multi
from vi_rnn.saving import load_model

%matplotlib inline

In [ ]:
hostname = socket.gethostname()
print("hostname:", hostname)

if hostname == "MatthijsDesktop":
    repo = Path("/home/matthijs/swm_rnn")
else:
    repo = Path.cwd().parent

path = str(repo / "data") + "/"
cache_path = repo / "data" / "processed" / "df_rank_delay_tuning.pkl"
decode_cache = repo / "data" / "processed" / "df_rank_within_decoding.pkl"
decode_pkls = {
    16: repo / "data" / "processed_r16" / "df_cross_decoding_data.pkl",
    32: repo / "data" / "processed_r32" / "df_cross_decoding_data.pkl",
    64: repo / "data" / "processed" / "df_cross_decoding_data.pkl",
}

rank_dirs = {
    16: repo / "final_models" / "rank_16",
    32: repo / "final_models" / "rank_32",
    64: repo / "final_models" / "macaque",
}
n_models = 10
model_dirs = {
    16: [
        "models_SWM_low_rank_one_to_one_dim_z_16_date_2026_09_19_T_00_51_48",
        "models_SWM_low_rank_one_to_one_dim_z_16_date_2026_09_19_T_13_52_07",
        "models_SWM_low_rank_one_to_one_dim_z_16_date_2026_09_19_T_21_03_57",
        "models_SWM_low_rank_one_to_one_dim_z_16_date_2026_09_20_T_16_43_53",
        "models_SWM_low_rank_one_to_one_dim_z_16_date_2026_09_21_T_00_31_13",
        "models_SWM_low_rank_one_to_one_dim_z_16_date_2026_09_22_T_02_00_56",
        "models_SWM_low_rank_one_to_one_dim_z_16_date_2026_09_22_T_19_35_14",
        "models_SWM_low_rank_one_to_one_dim_z_16_date_2026_09_23_T_19_09_04",
        "models_SWM_low_rank_one_to_one_dim_z_16_date_2026_09_24_T_04_19_40",
        "models_SWM_low_rank_one_to_one_dim_z_16_date_2026_09_24_T_17_59_39",
    ][:n_models],
    32: [
        "models_SWM_low_rank_one_to_one_dim_z_32_date_2026_09_12_T_21_50_03",
        "models_SWM_low_rank_one_to_one_dim_z_32_date_2026_09_13_T_05_42_27",
        "models_SWM_low_rank_one_to_one_dim_z_32_date_2026_09_13_T_14_58_56",
        "models_SWM_low_rank_one_to_one_dim_z_32_date_2026_09_14_T_00_05_40",
        "models_SWM_low_rank_one_to_one_dim_z_32_date_2026_09_14_T_22_49_32",
        "models_SWM_low_rank_one_to_one_dim_z_32_date_2026_09_15_T_19_07_22",
        "models_SWM_low_rank_one_to_one_dim_z_32_date_2026_09_16_T_04_28_45",
        "models_SWM_low_rank_one_to_one_dim_z_32_date_2026_09_16_T_16_00_02",
        "models_SWM_low_rank_one_to_one_dim_z_32_date_2026_09_16_T_23_49_32",
        "models_SWM_low_rank_one_to_one_dim_z_32_date_2026_09_17_T_18_30_27",
    ][:n_models],
    64: [
        "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_28_T_05_34_01",
        "SWM_low_rank_one_to_one_dim_z_64_date_2026_05_01_T_22_03_56",
        "SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_18_03_12",
        "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_17_02_13",
        "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_30_36",
        "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_26_38",
        "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_27_25",
        "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_30_22",
        "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_20_35",
        "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_13_05",
    ][:n_models],
}

In [ ]:
# --- controls (match notebook 08 stimulus-response cell) ---
seq_len = 3  # 45–65 bins only line up on length-3 trials
t1_stim = 45
t2_stim = 65
run = False
run_decode = False


In [ ]:
def macaque_from_sessions(sessions):
    s0 = sessions[0]
    if "ocean" in s0:
        return "ocean"
    if "groot" in s0:
        return "groot"
    return s0[5:10]


if run:
    rows = []
    for rank, names in model_dirs.items():
        out_dir = rank_dirs[rank]
        for name in names:
            model_dir = out_dir / name
            print(f"\nrank {rank}  {name}")
            vae, training_params, task_params = load_model(
                str(model_dir), load_encoder=True, backward_compat=False
            )
            task_params["path"] = path
            macaque = macaque_from_sessions(task_params["sessions"])
            task = SWM_dataset_multi(task_params)
            session_ids = list(
                getattr(task, "model_session_ids", range(len(task.sessions)))
            )
            for model_sess_id in session_ids:
                row = per_session_delay_tuning(
                    vae,
                    task,
                    model_sess_id,
                    seq_len=seq_len,
                    t1=t1_stim,
                    t2=t2_stim,
                )
                if row is None:
                    continue
                print(
                    f"  sess {model_sess_id}: n={row['n_trials']} "
                    f"r_tuning={row['r_tuning']:.3f} (tv={row['r_tuning_tv']:.3f})"
                )
                row = dict(row)
                row["rank"] = int(rank)
                row["name"] = name
                row["macaque"] = macaque
                rows.append(row)
    df = pd.DataFrame(rows)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    pickle.dump(df, open(cache_path, "wb"))
else:
    df = pickle.load(open(cache_path, "rb"))

print(df.head())
print(df.groupby(["rank", "macaque"]).size())

In [ ]:
# one row per session: mean over models, then plots spread over sessions
df_session = (
    df.groupby(["rank", "session", "macaque"], as_index=False)
    .mean(numeric_only=True)
)

print("Per-session means (models averaged), then mean/std over sessions:")
print(
    df_session.groupby(["rank", "macaque"])[["r_tuning", "r_tuning_tv"]]
    .agg(["mean", "std"])
    .round(3)
)


In [ ]:
macaque_labels = {"groot": "macaque g", "ocean": "macaque o"}
for macaque, sub in df_session.groupby("macaque"):
    plot_metric_by_rank(
        sub,
        value_col="r_tuning",
        baseline_df=sub,
        baseline_col="r_tuning_tv",
        ylabel="delay tuning $r$",
        title=macaque_labels.get(macaque, macaque),
        ylims=(0, 1),
        save_path=f"../paper_figures/rank_delay_tuning_r_{macaque}.pdf",
    )


## Within-session decoding

Linear SVM on delay latents (notebook 01), train and test within the same session. Chance is $1/6$. Same aggregation as tuning: mean over models, spread over sessions.


In [ ]:
if run_decode:
    rows_dec = []
    for rank, pkl in decode_pkls.items():
        df_z = pickle.load(open(pkl, "rb"))
        for _, row in df_z.iterrows():
            acc = within_session_decoding_acc(
                row["Z_train"],
                row["labels_train"],
                row["Z_test"],
                row["labels_test"],
            )
            for sess_i, acc_s in enumerate(acc):
                rec = {
                    "rank": int(rank),
                    "name": row["name"],
                    "macaque": row["macaque"],
                    "session": int(sess_i),
                    "acc": float(np.mean(acc_s)),
                    "acc_p1": float(acc_s[0]),
                    "acc_p2": float(acc_s[1]),
                    "acc_p3": float(acc_s[2]),
                }
                rows_dec.append(rec)
    df_dec = pd.DataFrame(rows_dec)
    decode_cache.parent.mkdir(parents=True, exist_ok=True)
    pickle.dump(df_dec, open(decode_cache, "wb"))
else:
    df_dec = pickle.load(open(decode_cache, "rb"))

df_dec_session = (
    df_dec.groupby(["rank", "session", "macaque"], as_index=False)
    .mean(numeric_only=True)
)
print(df_dec.groupby(["rank", "macaque"]).size())
print(
    df_dec_session.groupby(["rank", "macaque"])[["acc", "acc_p1", "acc_p2", "acc_p3"]]
    .agg(["mean", "std"])
    .round(3)
)


In [ ]:
chance = 1 / 6
macaque_labels = {"groot": "macaque g", "ocean": "macaque o"}
for macaque, sub in df_dec_session.groupby("macaque"):
    plot_metric_by_rank(
        sub,
        value_col="acc",
        ylabel="accuracy",
        title=macaque_labels.get(macaque, macaque),
        ylims=(0, 1),
        href=chance,
        save_path=f"../paper_figures/rank_decode_acc_{macaque}.pdf",
    )
